# 02 - 数据流与 Collate 函数

本 notebook 讲解 MiniMind-O 的数据侧：
1. `OmniDataset` 如何从 parquet 加载多轮对话、音频、图像
2. prompt 构造与 tokenizer 行为
3. label 生成（只训练最后一个 assistant）
4. audio labels 的 8 层 RVQ 对齐
5. `omni_collate_fn` 如何处理变长输入
6. 数据增强与 scheduled sampling

对应源码：`dataset/omni_dataset.py`。

## 1. 数据集输入格式

训练数据是 parquet 格式，关键列：
- `conversations`：JSON 字符串，多轮对话 `[{role, content}, ...]`
- `question_audios`：每轮 user 对应的音频 bytes list（可为空）
- `answer_audios`：每轮 assistant 对应的音频 codes list
- `image_bytes`：图片 bytes list
- `ref_audios`：参考音频 codes（音色克隆）
- `spk_emb`：说话人嵌入（192 维）

先检查是否有示例数据可用。

In [ ]:
import os, sys, glob, json
ROOT = "/home/genesis/Projects/minimind-o"
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# 查找可能的示例 parquet
candidates = sorted(glob.glob("dataset/**/*.parquet", recursive=True))[:20]
print(f"找到 {len(candidates)} 个 parquet 文件")
for p in candidates:
    size = os.path.getsize(p) / 1024**2
    print(f"  {p} ({size:.1f} MB)")

## 2. OmniDataset 初始化

核心参数：
- `data_path`：逗号分隔的 parquet 路径
- `tokenizer`：transformers AutoTokenizer
- `audio_processor / vision_processor`：来自 SenseVoice / SigLIP2
- `max_length`：最大序列长度
- `scheduled_sampling`： scheduled sampling 概率

初始化时会一次性把 parquet 读到内存（`pa.concat_tables`），因此对内存要求较高。

In [ ]:
import torch
from transformers import AutoTokenizer
from dataset.omni_dataset import OmniDataset

# 使用项目 tokenizer
tokenizer = AutoTokenizer.from_pretrained("model")
print(f"词表大小: {len(tokenizer)}")
print(f"special tokens: {tokenizer.special_tokens_map}")

# 观察特殊 token id
for tok in ["<|audio_pad|>", "<|image_pad|>", "assistant\n", "user\n"]:
    ids = tokenizer.encode(tok, add_special_tokens=False)
    print(f"{tok!r} -> {ids}")

## 3. prompt 构造流程

`create_chat_prompt` 做三件事：
1. `pre_processing_chat`：20% 概率加 system prompt。
2. 对最后一个 user：随机决定 audio token 与文本的拼接方式。
3. 对 `<image>` 占位符：随机决定图片 token 位置。
4. `post_processing_chat`：以 80% 概率移除 `<think>...</think>` 思考标记。

注意：`create_chat_prompt` 接受 `audio_features_length`，用于把音频特征长度转成对应数量的 `<|audio_pad|>` token。

In [ ]:
# 构造一个简单对话观察 prompt
conversations = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！有什么可以帮你的吗？"},
    {"role": "user", "content": "介绍一下你自己"},
]

# 为了调用 create_chat_prompt，先构造一个 dummy dataset
ds = OmniDataset(
    data_path="dataset/_calib/sft_t2a_144rows.parquet" if os.path.exists("dataset/_calib/sft_t2a_144rows.parquet") else "dataset/eval_omni/nonexistent.parquet",
    tokenizer=tokenizer,
    max_length=512,
)

prompt = ds.create_chat_prompt(conversations, audio_features_length=0)
print(prompt)
print(f"\ntokenized length: {len(tokenizer(prompt).input_ids)}")

## 4. label 生成：只训练最后一个 assistant

`generate_text_labels` 扫描 `bos_id + assistant` 到 `eos_id` 的区间，把中间部分设为 label。
然后 `__getitem__` 会把除最后一个 assistant 以外的 label 全部 mask 成 -100。

这意味着：
- 所有 system / user prompt 都不计算 loss。
- 只有最后一个 assistant 的回答参与文本 loss。
- 多轮对话中前面的 assistant 回答被忽略。

In [ ]:
# 观察 bos_id / eos_id
print(f"bos_id: {ds.bos_id}")
print(f"eos_id: {ds.eos_id}")
print(f"think_end_ids: {ds.think_end_ids}")

input_ids = tokenizer(prompt).input_ids[:ds.max_length]
labels, ranges = ds.generate_text_labels(input_ids)
print(f"assistant ranges: {ranges}")

# 打印前 40 个 token 的 label
for i in range(min(40, len(input_ids))):
    tok = tokenizer.decode([input_ids[i]])
    lab = labels[i]
    print(f"{i:3d} {tok!r:15s} label={lab}")

## 5. audio labels 的 8 层 RVQ 对齐

`answer_audios` 中的 codes 是 flatten 的 `[layer0, layer1, ..., layer7, layer0, layer1, ...]`。
`__getitem__` 按 step 解包成 8 层，每层末尾追加 `audio_stop_token`（2050）。

目标位置：
- `assistant_start + layer_idx + 1` 开始填充第 `layer_idx` 层 code。
- 即 layer0 从 `assistant_start+1` 开始，layer1 从 `assistant_start+2` 开始，依次延迟。

这与 `stream_generate` 中 audio 比 text 延迟 1 step 的设计一致。

In [ ]:
# 构造 dummy audio codes 观察对齐
dummy_codes = list(range(8)) * 5  # 5 帧
audio_codes_8layers = [[] for _ in range(8)]
for i in range(0, len(dummy_codes) - 7, 8):
    for j in range(8):
        audio_codes_8layers[j].append(dummy_codes[i + j])
for layer in audio_codes_8layers:
    layer.append(2050)  # stop token

for i, layer in enumerate(audio_codes_8layers):
    print(f"layer {i}: {layer}")

## 6. spk_emb 与 ref_codes 注入

音色克隆数据流：
- `spk_emb`：192 维说话人嵌入，放在 assistant 回答的第一个位置前一个 token。
- `ref_codes`：参考音频 codes，右对齐填充到 assistant 回答前。
- `has_ref` 有 50% 概率 drop，只保留 spk_emb。

代码见 `dataset/omni_dataset.py:292-310`。

## 7. omni_collate_fn

代码见 `trainer/train_sft_omni.py:218-243`。

batch 组装逻辑：
- `input_ids / labels / audio_labels / spk_emb`：直接 stack，要求 dataset 已经 pad 到 `max_length`。
- `audio_inputs`：对有效音频按最大时间维度 pad 后 cat。
- `pixel_values`：如果是 dict（SigLIP2 输出），按 key cat；否则直接 stack。

**注意**：
- dataset 中的 dummy audio `torch.zeros(1,1,560)` 会被 `omni_collate_fn` 识别为有效音频（非 None），参与 pad/cat。
- `encode_audio_inputs` 中用 `any(1)` 过滤全 0 样本，因此 dummy 会被跳过。

In [ ]:
# 手动构造一个 batch 观察 collate
from trainer.train_sft_omni import omni_collate_fn

T = 32
batch = [
    (
        torch.randint(0, 6400, (9, T)),   # input_ids
        torch.randint(-100, 6400, (T,)),    # labels
        torch.randint(-100, 2112, (8, T)),  # audio_labels
        torch.randn(10, 560) if i % 2 == 0 else None,  # audio_inputs
        10 if i % 2 == 0 else 0,            # audio_lens
        torch.randn(3, 256, 256) if i % 3 == 0 else None,  # pixel_values
        torch.randn(192),                   # spk_emb
    )
    for i in range(4)
]

ids, labels, audio_labels, audio_inputs, audio_lens, pixel_values, spk_emb = omni_collate_fn(batch)
print(f"input_ids shape: {ids.shape}")
print(f"audio_inputs shape: {audio_inputs.shape if audio_inputs is not None else None}")
print(f"pixel_values shape: {pixel_values.shape if not hasattr(pixel_values, 'keys') else {k:v.shape for k,v in pixel_values.items()}}")
print(f"audio_lens: {audio_lens}")

## 8. 数据增强

`augment_wav` 实现了丰富的音频增强：
- 变速（0.7~1.6x）
- 高斯噪声
- 音量缩放
- 时间遮蔽
- 低通滤波
- 混响
- 粉红噪声

`augment_mel` 对 SenseVoice fbank 做 SpecAugment 频率/时间遮蔽。

**注意**：
- 所有增强都在 `__getitem__` 中动态进行，CPU 侧耗时较大，是训练吞吐的一个瓶颈。
- 如果追求速度，可以考虑离线预计算 audio features 到 parquet。

## 9. Scheduled Sampling

`apply_scheduled_sampling` 以 5% 概率把 audio/text 输入替换为随机值，让模型学会从错误历史中恢复。

代码见 `dataset/omni_dataset.py:199-209`。

**注意**：
- 只替换 input_ids，不替换 labels，因此 loss 仍然用 GT 监督。
- 默认概率 0.05，比较小；增大可能让训练更不稳定。
- 实现中通过 `audio_labels != -100` 找 audio 区域，然后用 `randint` 替换，注意 randint 范围是 `[0, audio_vocab_size)`，包含特殊 token 区域，可能引入非法 code。

## 10. 数据侧问题清单

| 问题 | 位置 | 影响 | 建议 |
| --- | --- | --- | --- |
| parquet 全量加载到内存 | `omni_dataset.py:54` | 大表启动慢、内存占用高 | 可按 shard / 流式读取 |
| 音频增强全在 CPU 动态做 | `augment_wav/mel` | 数据加载瓶颈 | 可离线预计算 fbank |
| dummy audio 是 `zeros(1,1,560)` | `omni_dataset.py:251` | collate 后仍参与编码器 forward，靠 `any(1)` 过滤 | 注意 batch_mask 逻辑 |
| 多轮对话只训练最后 assistant | `omni_dataset.py:278-280` | 前面的 assistant 回答不被学习 | 多轮能力受限 |
| scheduled sampling 随机到特殊 token | `omni_dataset.py:205,208` | 可能让模型看到非法 audio/text code | 可限制随机范围 |
| `audio_inputs` 和 `pixel_values` 在 dataset 中不能为 None | `omni_dataset.py:250-254` | 必须提供 dummy tensor，增加 collate 复杂度 | 历史设计，需理解 |